# Первые эксперименты с извлечением тезисов из текста

Лаба 1. Первый прогон сделан вручную и описан честно. Ниже добавлен воспроизводимый
вызов Gemini API с фиксированными параметрами; секрет берётся только из переменной окружения.

Полный разбор результатов и выводы: docs/experiments/lab1.md

## Тестовые фрагменты методичек

Три придуманных коротких примера позволяют повторить сравнение без публикации чужих материалов.

In [ ]:
test_fragments = {
    "cache": ("Кэш процессора — быстрая память между процессором и оперативной памятью. "
              "Кэш делится на уровни L1, L2 и L3; каждый следующий уровень медленнее, "
              "но больше. При cache miss процессор обращается к оперативной памяти."),
    "network": ("TCP устанавливает соединение через three-way handshake и гарантирует "
                "порядок доставки. UDP не устанавливает соединение и не гарантирует доставку."),
    "formula": ("Закон Ома: I = U / R, где I — ток, U — напряжение, R — сопротивление. "
                "При постоянном R удвоение U приводит к удвоению I."),
}
test_fragment = test_fragments["cache"]
test_fragments

## Воспроизводимый запуск через API

Перед запуском задайте `GEMINI_API_KEY`. Модель по умолчанию — `gemini-3.8-flash`,
температура — 0.2. Ключ и результаты не записываются в репозиторий автоматически.

In [ ]:
import json
import os
import urllib.parse
import urllib.request

MODEL = os.getenv("LLM_MODEL_NAME", "gemini-3.8-flash")
TEMPERATURE = 0.2

def call_gemini(prompt):
    api_key = os.environ["GEMINI_API_KEY"]
    url = ("https://generativelanguage.googleapis.com/v1beta/models/"
           f"{MODEL}:generateContent?key={urllib.parse.quote(api_key)}")
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": TEMPERATURE},
    }
    request = urllib.request.Request(
        url, data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST"
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        body = json.load(response)
    return body["candidates"][0]["content"]["parts"][0]["text"]


In [ ]:
def build_prompts(fragment):
    simple = "Вот текст методички. Составь вопросы для повторения по нему.\n\n" + fragment
    structured = (
        "Ты помогаешь студенту готовиться к устной защите лабораторной работы. "
        f"Вот текст методички: {fragment}. Выдели от 5 до 10 ключевых тезисов. "
        "Для каждого тезиса сформулируй короткий вопрос и эталонный ответ. "
        "Не добавляй факты, которых нет в тексте."
    )
    return {"simple": simple, "structured": structured}

if "GEMINI_API_KEY" not in os.environ:
    print("Задайте GEMINI_API_KEY и повторно выполните эту ячейку")
    results = []
else:
    results = []
    for case_name, fragment in test_fragments.items():
        for prompt_name, prompt in build_prompts(fragment).items():
            results.append({
                "case": case_name, "prompt": prompt_name,
                "model": MODEL, "temperature": TEMPERATURE,
                "output": call_gemini(prompt),
            })
results

Оцените каждый результат по трём признакам: число конкретных вопросов, отсутствие фактов
вне исходного фрагмента и соблюдение формата «вопрос — эталон». Дата, модель и температура
сохраняются в объекте результата, поэтому прогон можно сравнить с последующим.

## Промпт 1, простой без структуры

Результат вручную: модель выдала 3 общих вопроса вроде "Что такое кэш процессора?".
Слишком похоже на пересказ определения, не годится для проверки понимания.

In [ ]:
prompt_v1 = (
    "Вот текст методички. Составь вопросы для повторения по нему.\n\n"
    + test_fragment
)
print(prompt_v1)

## Промпт 2, с ролью и форматом вопрос-эталон

Результат вручную заметно лучше, вопросы конкретные и проверяемые:

- Зачем нужен кэш процессора, а не только оперативная память?
- Что происходит при промахе кэша?
- Чем отличаются уровни L1, L2 и L3?

In [ ]:
prompt_v2 = (
    "Ты помогаешь студенту готовиться к устной защите лабораторной работы. "
    "Вот текст методички: {fragment}. Выдели от 5 до 10 ключевых тезисов. "
    "Для каждого тезиса сформулируй короткий вопрос, на который можно ответить "
    "устно за 20-30 секунд, и короткий эталонный ответ. Не задавай вопросы по "
    "вещам, которых нет в тексте."
).format(fragment=test_fragment)
print(prompt_v2)

## Вывод

Промпт с явной ролью и ограничением "не придумывай то чего нет в тексте" даёт более
конкретные вопросы. Берём его за основу подхода, полное обоснование выбора
в docs/research.md, раздел Решение.